# AMS — Protocolo estatístico sobre o AI4I2020

**Disciplina 3 — Matemática e Estatística para IA · CEDAGI/UTFPR**

Dataset: *AI4I 2020 Predictive Maintenance Dataset* (UCI, ID 601) — o mesmo
dataset real da D1 (`data/raw/ai4i2020.csv`). O `dominio.md` do projeto diz
que "da D3 em diante, [o AMS] ganha protocolo estatístico" — é isso que este
notebook entrega.

**Diferença para os outros dois lugares que tocam neste dataset**, pra não
confundir:
- `notebooks/01-eda.ipynb` (D1) já respondeu 8 perguntas básicas sobre o
  dataset (taxa de falha, distribuição de desgaste, torque×rotação, limiar
  de HDF, frequência dos modos, falhas sem modo, regra "desgaste > X",
  correlação). Este notebook **não repete** essas perguntas — parte delas
  e aprofunda: forma da distribuição, testes de hipótese, outliers formais,
  comparação de grupos, e uma pergunta nova no fim.
- `D03/D3-03-estatistica-descritiva/` é um laboratório **separado e
  obrigatório** da disciplina: gera `relatorio-dado.md` (texto, sem
  gráfico) com um script fechado (`medidas.py`), validado por `teste.py`.
  Este notebook não substitui aquela entrega.

**Regra herdada do 01-eda:** nenhum gráfico ou teste existe por si só — cada
seção é uma pergunta, respondida com código, gráfico/teste e uma frase de
**Conclusão** em negrito. Este notebook tem uma segunda saída: um HTML
interativo (`02-protocolo-estatistico.html`, gerado por
`notebooks/extra-pessoal/gerar_html_protocolo_estatistico.py`) com os mesmos gráficos de
comparação, em Plotly, pra manipular sem reabrir o notebook.

**Os números não ficam fixos neste notebook.** As células de "Conclusão" abaixo apontam para o relatório gerado por `notebooks/extra-pessoal/gerar_relatorio_protocolo_estatistico.py` — um `.md` por fonte de dado (`02-protocolo-estatistico-amostra.md`, `02-protocolo-estatistico-raw.md`), com a narrativa recalculada a cada rodada. Troque `FONTE_DADOS` na célula abaixo à vontade: os gráficos deste notebook acompanham a troca, e o relatório da fonte escolhida é que tem a análise por extenso.

**Esta pasta (`notebooks/extra-pessoal/`) é produção própria, fora do fluxo de entrega**: o fluxo de `entrega/D0X_EY` só traz o que é copiado explicitamente para `main` (ver `CLAUDE.md` da raiz) — nada aqui vai junto por padrão.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from ams import analise
from ams.dados import RAIZ_DO_PACOTE, carregar, preparar
from ams.gerador_casos import (
    HDF_LIMITE_DELTA,
    HDF_LIMITE_ROTACAO,
    MARGEM_DIFICIL_HDF_DELTA,
    MARGEM_DIFICIL_HDF_ROTACAO,
    MARGEM_DIFICIL_OSF,
    MARGEM_DIFICIL_PWF,
    OSF_LIMITES,
    PWF_FAIXA_SEGURA,
    TWF_JANELA,
)

sns.set_theme(style="whitegrid")
COR_NAO_FALHA, COR_FALHA = "#4c72b0", "#c44e52"  # mesma paleta do 01-eda

# Troque para "amostra" para rodar com as 200 linhas versionadas (sem precisar do dataset completo).
FONTE_DADOS = "raw"  # "raw" (10.000 linhas, data/raw/) ou "amostra" (200 linhas, data/amostra/)
# FONTE_DADOS = "amostra"  # "raw" (10.000 linhas, data/raw/) ou "amostra" (200 linhas, data/amostra/)

caminho_dados = RAIZ_DO_PACOTE / "data" / ("raw" if FONTE_DADOS == "raw" else "amostra") / "ai4i2020.csv"

dataset = carregar(caminho=caminho_dados)
df = preparar(dataset.quadro)
df["potencia_w"] = df["potencia_kw"] * 1000.0          # PWF_FAIXA_SEGURA está em W
df["osf_produto"] = df["desgaste"] * df["torque"]       # min·Nm — mesma fórmula do gerador_casos
df["osf_limite"] = df["tipo"].astype(str).map(OSF_LIMITES).astype(float)

print(f"fonte dos dados: {dataset.fonte}")
print(f"dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(3)

fonte dos dados: raw/ai4i2020.csv (10000 linhas)
dimensões: 10000 linhas x 20 colunas


,udi,product_id,tipo,temperatura_ar,temperatura_processo,rotacao,torque,desgaste,falha,twf,hdf,pwf,osf,rnf,delta_temperatura,potencia_kw,algum_modo,potencia_w,osf_produto,osf_limite
0,1,M14860,M,298.1,308.6,1551,42.8,0,False,False,False,False,False,False,10.5,6.951591,False,6951.590560,0.0,12000.0
1,2,L47181,L,298.2,308.7,1408,46.3,3,False,False,False,False,False,False,10.5,6.826723,False,6826.722724,138.9,11000.0
2,3,L47182,L,298.1,308.5,1498,49.4,5,False,False,False,False,False,False,10.4,7.749388,False,7749.387543,247.0,11000.0


---

## 1 — Visão geral e qualidade dos dados

Antes de qualquer medida: o dataset tem o que diz ter? Confirmar, não supor
— é a mesma régua que `D3-03/medidas.py` aplica.

In [4]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0]

print(f"linhas: {len(df)}  |  UDIs únicos: {df['udi'].nunique()}  |  duplicados: {df.duplicated().sum()}")
print(f"colunas com valor faltante: {len(faltantes)}" + (f" — {dict(faltantes)}" if len(faltantes) else " (nenhuma)"))
print("\ndistribuição de `tipo`:")
print(df["tipo"].value_counts())
df.describe(include="all").T

linhas: 10000  |  UDIs únicos: 10000  |  duplicados: 0
colunas com valor faltante: 0 (nenhuma)

distribuição de `tipo`:
tipo
L    6000
M    2997
H    1003
Name: count, dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
udi,10000.0,NaN,NaN,NaN,5000.5,2886.89568,1.0,2500.75,5000.5,7500.25,10000.0
product_id,10000,10000,M14860,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo,10000,3,L,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperatura_ar,10000.0,NaN,NaN,NaN,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
temperatura_processo,10000.0,NaN,NaN,NaN,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
rotacao,10000.0,NaN,NaN,NaN,1538.7761,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
torque,10000.0,NaN,NaN,NaN,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
desgaste,10000.0,NaN,NaN,NaN,107.951,63.654147,0.0,53.0,108.0,162.0,253.0
falha,10000,2,False,9661,NaN,NaN,NaN,NaN,NaN,NaN,NaN
twf,10000,2,False,9954,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Conclusão:** ver seção 1 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 2 — Estatística descritiva univariada

Pra cada variável numérica: tendência central, dispersão e **forma** da
distribuição (assimetria e curtose) — o que o `describe()` sozinho não
mostra.

In [ ]:
numericas = ["temperatura_ar", "temperatura_processo", "rotacao", "torque",
             "desgaste", "delta_temperatura", "potencia_kw"]

resumo = pd.DataFrame({
    "media": df[numericas].mean(),
    "mediana": df[numericas].median(),
    "desvio_padrao": df[numericas].std(),
    "assimetria": df[numericas].skew(),
    "curtose": df[numericas].kurt(),
    "Q1": df[numericas].quantile(0.25),
    "Q3": df[numericas].quantile(0.75),
})
resumo["AIQ"] = resumo["Q3"] - resumo["Q1"]

# normaltest (D'Agostino K²) em vez de shapiro: shapiro degrada/avisa acima
# de ~5.000 amostras; normaltest não tem essa restrição.
resumo["normaltest_p"] = [stats.normaltest(df[c]).pvalue for c in numericas]
resumo.round(4)

In [ ]:
fig, eixos = plt.subplots(3, 3, figsize=(12, 9))
for eixo, coluna in zip(eixos.flat, numericas):
    sns.histplot(df[coluna], kde=True, ax=eixo, color=COR_NAO_FALHA)
    eixo.set_title(coluna)
    eixo.set_xlabel("")
for eixo in eixos.flat[len(numericas):]:
    eixo.axis("off")
fig.suptitle("Distribuição das variáveis numéricas")
plt.tight_layout()
plt.show()

**Conclusão:** ver seção 2 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 3 — Outliers via IQR

Mesma regra do boxplot que `D3-03/medidas.py` usa (fora de
`[Q1 − 1,5·AIQ, Q3 + 1,5·AIQ]`), agora com gráfico — e com a pergunta que
`medidas.py` deixa pro aluno responder manualmente: esses extremos são
ruído ou sinal?

In [ ]:
limite_inferior = resumo["Q1"] - 1.5 * resumo["AIQ"]
limite_superior = resumo["Q3"] + 1.5 * resumo["AIQ"]

outliers = pd.DataFrame(index=numericas)
outliers["n_extremos"] = [
    int(((df[c] < limite_inferior[c]) | (df[c] > limite_superior[c])).sum())
    for c in numericas
]
outliers["pct_extremos"] = (outliers["n_extremos"] / len(df) * 100).round(2)
outliers["pct_extremos_que_falharam"] = [
    round(df.loc[(df[c] < limite_inferior[c]) | (df[c] > limite_superior[c]), "falha"].mean() * 100, 1)
    if outliers.loc[c, "n_extremos"] > 0 else float("nan")
    for c in numericas
]
outliers

In [ ]:
fig, eixos = plt.subplots(3, 3, figsize=(12, 9))
for eixo, coluna in zip(eixos.flat, numericas):
    sns.boxplot(y=df[coluna], ax=eixo, color=COR_NAO_FALHA)
    eixo.set_title(coluna)
    eixo.set_ylabel("")
for eixo in eixos.flat[len(numericas):]:
    eixo.axis("off")
fig.suptitle("Outliers (regra do boxplot, mesma do D3-03)")
plt.tight_layout()
plt.show()

**Conclusão:** ver seção 3 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 4 — Correlação bivariada

O 01-eda já mediu torque×rotação (pergunta 3). Aqui a matriz completa, com
Spearman ao lado de Pearson — útil porque `desgaste` não é normal (seção 2).

In [ ]:
pearson = df[numericas].corr(method="pearson")
spearman = df[numericas].corr(method="spearman")

fig, (eixo1, eixo2) = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(pearson, annot=True, fmt=".2f", cmap="vlag", center=0, ax=eixo1, cbar=False)
eixo1.set_title("Pearson")
sns.heatmap(spearman, annot=True, fmt=".2f", cmap="vlag", center=0, ax=eixo2)
eixo2.set_title("Spearman")
plt.tight_layout()
plt.show()

print(f"torque x rotação (Pearson, via analise.py): {analise.correlacao_torque_rotacao(df):.3f}")

**Conclusão:** ver seção 4 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 5 — Falha vs. não-falha, por variável

Comparação de grupo com teste de hipótese: as distribuições de quem falhou
e quem não falhou são estatisticamente diferentes — e o quanto isso importa
na prática (não só o p-valor).

In [ ]:
variaveis_grupo = ["desgaste", "delta_temperatura", "torque", "potencia_kw"]

fig, eixos = plt.subplots(1, 4, figsize=(15, 4.5))
for eixo, coluna in zip(eixos, variaveis_grupo):
    sns.boxplot(data=df, x="falha", y=coluna, hue="falha", legend=False, ax=eixo,
                palette={False: COR_NAO_FALHA, True: COR_FALHA})
    eixo.set_xticks([0, 1])
    eixo.set_xticklabels(["não falhou", "falhou"])
    eixo.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
linhas = []
for coluna in variaveis_grupo:
    grupo_0 = df.loc[~df["falha"], coluna]
    grupo_1 = df.loc[df["falha"], coluna]
    estatistica_u, p_valor = stats.mannwhitneyu(grupo_1, grupo_0, alternative="two-sided")
    # tamanho de efeito rank-biserial: 1 indica separação perfeita, 0 nenhuma
    efeito_rank_biserial = 1 - (2 * estatistica_u) / (len(grupo_0) * len(grupo_1))
    linhas.append({
        "variavel": coluna,
        "mediana_nao_falhou": grupo_0.median(),
        "mediana_falhou": grupo_1.median(),
        "mannwhitney_p": p_valor,
        "efeito_rank_biserial": round(efeito_rank_biserial, 3),
    })
pd.DataFrame(linhas)

**Conclusão:** ver seção 5 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 6 — Tipo de produto × falha

O 01-eda já mostrou a taxa de falha por tipo (pergunta 1). Aqui, o teste de
associação formal — e se o tamanho do efeito acompanha a significância.

In [ ]:
tabela_contingencia = pd.crosstab(df["tipo"], df["falha"])
qui2, p_valor, graus_liberdade, esperado = stats.chi2_contingency(tabela_contingencia)

n = tabela_contingencia.to_numpy().sum()
menor_dimensao = min(tabela_contingencia.shape) - 1
cramers_v = (qui2 / (n * menor_dimensao)) ** 0.5

print(tabela_contingencia)
print(f"\nqui-quadrado: {qui2:.2f}  |  p-valor: {p_valor:.2e}  |  Cramér's V: {cramers_v:.3f}")
print(f"taxa de falha por tipo (%):\n{analise.taxa_falha_por_tipo(df).round(2)}")

**Conclusão:** ver seção 6 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 7 — Os cinco mecanismos de falha, quantificados

O 01-eda visualizou o limiar de HDF sozinho (pergunta 4). Aqui, os
limiares oficiais dos cinco modos — **os mesmos de `ams.gerador_casos`**,
importados, não reescritos — testados como REGRA preditiva contra a flag
real de cada modo. TWF é probabilístico dentro da janela (não todo caso na
zona falha); RNF é 0,1% aleatório e não tem variável de sensor que o preveja
— isso importa para a seção 8.

In [ ]:
previsto_twf = df["desgaste"].between(*TWF_JANELA)
previsto_hdf = (df["delta_temperatura"] < HDF_LIMITE_DELTA) & (df["rotacao"] < HDF_LIMITE_ROTACAO)
previsto_pwf = ~df["potencia_w"].between(*PWF_FAIXA_SEGURA)
previsto_osf = df["osf_produto"] >= df["osf_limite"]

def recall_precisao(previsto, real):
    verdadeiro_positivo = int((previsto & real).sum())
    recall = verdadeiro_positivo / real.sum() if real.sum() else float("nan")
    precisao = verdadeiro_positivo / previsto.sum() if previsto.sum() else float("nan")
    return recall, precisao, int(real.sum()), int(previsto.sum())

linhas = []
for nome, previsto, real in [
    ("TWF (zona de desgaste 200-240min)", previsto_twf, df["twf"]),
    ("HDF (delta<8,6K e rotação<1380rpm)", previsto_hdf, df["hdf"]),
    ("PWF (potência fora de 3500-9000W)", previsto_pwf, df["pwf"]),
    ("OSF (desgaste x torque >= limite por tipo)", previsto_osf, df["osf"]),
]:
    recall, precisao, n_real, n_previsto = recall_precisao(previsto, real)
    linhas.append({"mecanismo": nome, "recall": round(recall, 3), "precisao": round(precisao, 3),
                    "casos_reais": n_real, "casos_previstos_pela_regra": n_previsto})

print(f"RNF: {int(df['rnf'].sum())} casos reais em {len(df)} ({df['rnf'].mean()*100:.3f}%) "
      "— nenhuma variável de sensor prevê isso, é ruído puro por definição.")
pd.DataFrame(linhas)

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(12, 9))

eixos[0, 0].hist(df["desgaste"], bins=40, color=COR_NAO_FALHA)
eixos[0, 0].axvspan(*TWF_JANELA, color=COR_FALHA, alpha=0.2, label="janela TWF")
eixos[0, 0].set_title("TWF — desgaste e a janela de risco")
eixos[0, 0].legend()

sns.scatterplot(data=df, x="rotacao", y="delta_temperatura", hue="hdf", ax=eixos[0, 1],
                 palette={False: COR_NAO_FALHA, True: COR_FALHA}, s=12, alpha=0.6)
eixos[0, 1].axhline(HDF_LIMITE_DELTA, color="gray", linestyle="--")
eixos[0, 1].axvline(HDF_LIMITE_ROTACAO, color="gray", linestyle="--")
eixos[0, 1].set_title("HDF — fronteira delta×rotação")

eixos[1, 0].hist(df["potencia_w"], bins=40, color=COR_NAO_FALHA)
eixos[1, 0].axvspan(*PWF_FAIXA_SEGURA, color=COR_NAO_FALHA, alpha=0.15, label="faixa segura")
eixos[1, 0].set_title("PWF — potência e a faixa segura")
eixos[1, 0].legend()

sns.scatterplot(data=df, x="osf_limite", y="osf_produto", hue="osf", ax=eixos[1, 1],
                 palette={False: COR_NAO_FALHA, True: COR_FALHA}, s=12, alpha=0.6)
eixos[1, 1].plot(sorted(df["osf_limite"].unique()), sorted(df["osf_limite"].unique()),
                  color="gray", linestyle="--")
eixos[1, 1].set_title("OSF — produto desgaste×torque vs. limite por tipo")

plt.tight_layout()
plt.show()

**Conclusão:** ver seção 7 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## 8 — Proposta de classificador agregado (baixo / médio / alto)

`agent-spec.md` §4 define recall/precisão da classe "alto" e a taxa de
"alto classificado como baixo" como métricas do agente — e marca as três
como **"valor de hoje: não medido"**. Esta seção mede, pela primeira vez,
o que uma regra puramente física (sem nenhum modelo treinado) entrega:

- **alto**: TWF, HDF, PWF ou OSF sinalizados (RNF fica de fora — é
  imprevisível por construção, seção 7).
- **médio**: nenhum limiar cruzado, mas dentro da MESMA margem que
  `ams.gerador_casos` já usa pra chamar um caso de "difícil" (perto da
  fronteira, sem passar dela).
- **baixo**: o resto.

In [ ]:
medio_hdf = (
    ((df["delta_temperatura"] >= HDF_LIMITE_DELTA) &
     (df["delta_temperatura"] < HDF_LIMITE_DELTA + MARGEM_DIFICIL_HDF_DELTA) &
     (df["rotacao"] < HDF_LIMITE_ROTACAO)) |
    ((df["rotacao"] >= HDF_LIMITE_ROTACAO) &
     (df["rotacao"] < HDF_LIMITE_ROTACAO + MARGEM_DIFICIL_HDF_ROTACAO) &
     (df["delta_temperatura"] < HDF_LIMITE_DELTA))
)
medio_pwf = (
    df["potencia_w"].between(PWF_FAIXA_SEGURA[0], PWF_FAIXA_SEGURA[0] + MARGEM_DIFICIL_PWF) |
    df["potencia_w"].between(PWF_FAIXA_SEGURA[1] - MARGEM_DIFICIL_PWF, PWF_FAIXA_SEGURA[1])
)
medio_osf = (df["osf_produto"] >= df["osf_limite"] - MARGEM_DIFICIL_OSF) & (df["osf_produto"] < df["osf_limite"])

alto = previsto_twf | previsto_hdf | previsto_pwf | previsto_osf
medio = ~alto & (medio_hdf | medio_pwf | medio_osf)

risco = pd.Series("baixo", index=df.index)
risco[medio] = "medio"
risco[alto] = "alto"
df["risco_previsto"] = risco

print(df["risco_previsto"].value_counts())
print("\ntaxa de falha real por nível de risco previsto (%):")
print((df.groupby("risco_previsto", observed=True)["falha"].mean() * 100).round(2))

In [ ]:
real_alto = df["falha"] & (df["risco_previsto"] == "alto")
recall_alto = real_alto.sum() / df["falha"].sum()
precisao_alto = real_alto.sum() / (df["risco_previsto"] == "alto").sum()
taxa_alto_para_baixo = (df["falha"] & (df["risco_previsto"] == "baixo")).sum() / df["falha"].sum()

print(f"recall da classe 'alto':        {recall_alto*100:.1f}%  (critério agent-spec.md §4: >= 80%)")
print(f"precisão da classe 'alto':      {precisao_alto*100:.1f}%  (critério agent-spec.md §4: >= 80%)")
print(f"taxa 'alto' classificado baixo: {taxa_alto_para_baixo*100:.1f}%  (critério agent-spec.md §4: <= 5%)")

fig, eixo = plt.subplots(figsize=(6, 4))
cores_risco = {"baixo": "#55a868", "medio": "#dd8452", "alto": "#c44e52"}  # status: bom/atenção/crítico
ordem_risco = ["baixo", "medio", "alto"]
taxas_por_risco = df.groupby("risco_previsto", observed=True)["falha"].mean().reindex(ordem_risco) * 100
eixo.bar(taxas_por_risco.index, taxas_por_risco.values,
         color=[cores_risco[c] for c in taxas_por_risco.index])
eixo.set_ylabel("taxa de falha real (%)")
eixo.set_title("Taxa de falha real, por nível de risco previsto pela regra")
plt.tight_layout()
plt.show()

**Conclusão:** ver seção 8 do relatório da fonte escolhida acima (`02-protocolo-estatistico-amostra.md` ou `02-protocolo-estatistico-raw.md`, gerado por `gerar_relatorio_protocolo_estatistico.py`) — números e narrativa completa ficam lá, não aqui, para não destoar quando `FONTE_DADOS` mudar.

---

## Síntese para o projeto

A síntese de cada rodada está no fim do relatório correspondente (`02-protocolo-estatistico-amostra.md` / `-raw.md`). Vale comparar os dois lado a lado, não só ler um: por exemplo, a variável que mais separa falha de não-falha na seção 5 **muda** entre as duas rodadas (`desgaste` na amostra de 200 linhas, `torque` no conjunto completo de 10.000) — um achado que só aparece comparando, não lendo uma rodada isolada. É esse tipo de comparação que a separação em dois relatórios permite estudar depois, sem reabrir o notebook.